#  Machine Learning Model Training & Evaluation

##  Objective
Train, evaluate, and compare Gradient Boosting (**XGBoost**, **LightGBM**), **Random Forest**, and **Ridge Regressor** models to predict daily state-level Aadhaar enrolments using engineered temporal, lag, rolling, and interaction features.

In [1]:
import os, glob, json, joblib, time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb

STATE_ALIASES = {
    "andaman and nicobar islands": "Andaman & Nicobar",
    "andaman & nicobar islands": "Andaman & Nicobar",
    "a & n islands": "Andaman & Nicobar",
    "andhra pradesh": "Andhra Pradesh", "arunachal pradesh": "Arunachal Pradesh",
    "assam": "Assam", "bihar": "Bihar", "chandigarh": "Chandigarh",
    "chhattisgarh": "Chhattisgarh", "chhatisgarh": "Chhattisgarh",
    "dadra and nagar haveli": "Dadra & Nagar Haveli and Daman & Diu",
    "daman and diu": "Dadra & Nagar Haveli and Daman & Diu",
    "dadra and nagar haveli and daman and diu": "Dadra & Nagar Haveli and Daman & Diu",
    "delhi": "Delhi", "nct of delhi": "Delhi", "goa": "Goa", "gujarat": "Gujarat",
    "haryana": "Haryana", "himachal pradesh": "Himachal Pradesh",
    "jammu and kashmir": "Jammu and Kashmir", "jammu & kashmir": "Jammu and Kashmir",
    "jharkhand": "Jharkhand", "karnataka": "Karnataka", "kerala": "Kerala",
    "ladakh": "Ladakh", "lakshadweep": "Lakshadweep",
    "madhya pradesh": "Madhya Pradesh", "maharashtra": "Maharashtra",
    "manipur": "Manipur", "meghalaya": "Meghalaya", "mizoram": "Mizoram",
    "nagaland": "Nagaland", "odisha": "Odisha", "orissa": "Odisha",
    "puducherry": "Puducherry", "pondicherry": "Puducherry", "punjab": "Punjab",
    "rajasthan": "Rajasthan", "sikkim": "Sikkim", "tamil nadu": "Tamil Nadu",
    "telangana": "Telangana", "tripura": "Tripura", "uttar pradesh": "Uttar Pradesh",
    "uttarakhand": "Uttarakhand", "uttaranchal": "Uttarakhand", "west bengal": "West Bengal"
}

def _norm(val):
    if pd.isna(val): return float('nan')
    return STATE_ALIASES.get(str(val).strip().lower(), str(val).strip().title())

def load_raw(data_dir="."):
    def _read(pattern):
        files = sorted(glob.glob(os.path.join(data_dir, pattern), recursive=True))
        dfs = [pd.read_csv(f, dtype={"state": str, "district": str}) for f in files]
        if not dfs: return pd.DataFrame()
        df = pd.concat(dfs, ignore_index=True)
        df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="coerce")
        df["norm_state"] = df["state"].apply(_norm)
        return df

    enrol = _read("api_data_aadhar_enrolment/**/*.csv")
    if not enrol.empty:
        enrol["total_enrolments"] = enrol[["age_0_5","age_5_17","age_18_greater"]].fillna(0).sum(axis=1)
        enrol = enrol.groupby(["date","norm_state"])[["age_0_5","age_5_17","age_18_greater","total_enrolments"]].sum().reset_index()

    demo = _read("api_data_aadhar_demographic/**/*.csv")
    if not demo.empty:
        demo["demo_total"] = demo[["demo_age_5_17","demo_age_17_"]].fillna(0).sum(axis=1)
        demo = demo.groupby(["date","norm_state"])[["demo_age_5_17","demo_age_17_","demo_total"]].sum().reset_index()

    bio = _read("api_data_aadhar_biometric/**/*.csv")
    if not bio.empty:
        bio["bio_total"] = bio[["bio_age_5_17","bio_age_17_"]].fillna(0).sum(axis=1)
        bio = bio.groupby(["date","norm_state"])[["bio_age_5_17","bio_age_17_","bio_total"]].sum().reset_index()

    m = pd.merge(enrol, demo, on=["date","norm_state"], how="outer") if not enrol.empty else demo
    if not bio.empty: m = pd.merge(m, bio, on=["date","norm_state"], how="outer")
    for c in ["total_enrolments","demo_total","bio_total"]:
        if c in m.columns: m[c] = m[c].fillna(0)
    return m

def build_features(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["norm_state","date"]).reset_index(drop=True)

    states = df["norm_state"].dropna().unique()
    all_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    grid = pd.MultiIndex.from_product([states, all_dates], names=["norm_state","date"]).to_frame(index=False)
    full = pd.merge(grid, df, on=["norm_state","date"], how="left")

    for c in ["age_0_5","age_5_17","age_18_greater","total_enrolments",
              "demo_age_5_17","demo_age_17_","demo_total","bio_age_5_17","bio_age_17_","bio_total"]:
        if c in full.columns: full[c] = full[c].fillna(0)

    # Calendar
    full["day_of_week"]      = full["date"].dt.dayofweek
    full["day_of_month"]     = full["date"].dt.day
    full["month"]            = full["date"].dt.month
    full["quarter"]          = full["date"].dt.quarter
    full["day_of_year"]      = full["date"].dt.dayofyear
    full["is_weekend"]       = full["day_of_week"].isin([5,6]).astype(int)
    full["days_since_start"] = (full["date"] - full["date"].min()).dt.days

    # Cyclical
    full["sin_dow"]   = np.sin(2*np.pi*full["day_of_week"]/7)
    full["cos_dow"]   = np.cos(2*np.pi*full["day_of_week"]/7)
    full["sin_month"] = np.sin(2*np.pi*full["month"]/12)
    full["cos_month"] = np.cos(2*np.pi*full["month"]/12)

    # Ordinal state code
    full["state_cat"] = full["norm_state"].astype("category").cat.codes

    dfs = []
    for state, grp in full.groupby("norm_state"):
        grp = grp.sort_values("date").copy()
        for lag in [1, 7, 14, 30]:
            grp[f"lag_{lag}"]      = grp["total_enrolments"].shift(lag)
            grp[f"bio_lag_{lag}"]  = grp["bio_total"].shift(lag)
            grp[f"demo_lag_{lag}"] = grp["demo_total"].shift(lag)
        for w in [7, 14, 30]:
            s = grp["total_enrolments"].shift(1)
            grp[f"rolling_mean_{w}"] = s.rolling(w, min_periods=1).mean()
            grp[f"rolling_std_{w}"]  = s.rolling(w, min_periods=1).std().fillna(0)
            grp[f"bio_rolling_{w}"]  = grp["bio_total"].shift(1).rolling(w, min_periods=1).mean()
            grp[f"demo_rolling_{w}"] = grp["demo_total"].shift(1).rolling(w, min_periods=1).mean()
        grp["bio_to_enrol_ratio"]  = (grp["bio_rolling_7"]  / (grp["rolling_mean_7"] + 1)).fillna(0)
        grp["demo_to_enrol_ratio"] = (grp["demo_rolling_7"] / (grp["rolling_mean_7"] + 1)).fillna(0)
        dfs.append(grp)

    return pd.concat(dfs, ignore_index=True).fillna(0)

raw_panel    = load_raw(data_dir=".")
processed_df = build_features(raw_panel)
print(f"Dataset shape: {processed_df.shape}")
print(f"Date range: {processed_df['date'].min().date()} -> {processed_df['date'].max().date()}")
print(f"States: {processed_df['norm_state'].nunique()}")


Dataset shape: (15912, 50)
Date range: 2025-03-01 -> 2025-12-31
States: 52


## ⚙️ Model Training & Chronological Evaluation

We split data chronologically into an 80% historical training set and a 20% recent testing set.
We train XGBoost, LightGBM, Random Forest, and Ridge models and compare their metrics (R², RMSE, MAE).

In [2]:
# ── FIRST-PRINCIPLES FIX: raw+scaled for Ridge+RF, log for XGB/LGB ─────────
import time
from sklearn.preprocessing import StandardScaler

target_col = "total_enrolments"
base_exclude = [
    "date","norm_state","state","district","pincode",
    "age_0_5","age_5_17","age_18_greater","total_enrolments",
    "demo_age_5_17","demo_age_17_","demo_total",
    "bio_age_5_17","bio_age_17_","bio_total"
]

# 1. Temporal split FIRST
unique_dates = sorted(processed_df["date"].unique())
split_date   = unique_dates[int(len(unique_dates) * 0.8)]
print(f"Train: up to {pd.Timestamp(split_date).date()} | Test: after")
print(f"Train rows: {(processed_df['date'] < split_date).sum()} | Test rows: {(processed_df['date'] >= split_date).sum()}")

train_df = processed_df[processed_df["date"] < split_date].copy()
test_df  = processed_df[processed_df["date"] >= split_date].copy()

# 2. State-level scale features computed STRICTLY on train (no leakage)
state_stats = (
    train_df.groupby("norm_state")["total_enrolments"]
    .agg(state_mean_enrol="mean", state_std_enrol="std", state_median_enrol="median")
    .reset_index()
)
state_stats["state_std_enrol"] = state_stats["state_std_enrol"].fillna(0)

train_df = train_df.merge(state_stats, on="norm_state", how="left")
test_df  = test_df.merge(state_stats, on="norm_state", how="left")
global_mean = float(train_df["total_enrolments"].mean())
for c in ["state_mean_enrol","state_std_enrol","state_median_enrol"]:
    test_df[c] = test_df[c].fillna(global_mean)
    train_df[c] = train_df[c].fillna(global_mean)

# 3. Log-transform target for gradient boosting only
train_df["target_log"] = np.log1p(train_df[target_col])
test_df["target_log"]  = np.log1p(test_df[target_col])

feature_cols = [c for c in train_df.columns if c not in base_exclude + ["target_log"]]
print(f"Feature count: {len(feature_cols)}")

X_train = train_df[feature_cols]
X_test  = test_df[feature_cols]
y_train_log = train_df["target_log"]
y_test_raw  = test_df[target_col]

# 4. StandardScaler for linear/tree models on raw target
scaler_raw = StandardScaler()
X_train_scaled = scaler_raw.fit_transform(X_train)
X_test_scaled  = scaler_raw.transform(X_test)

os.makedirs("pkl_models", exist_ok=True)
results = []
best_model_obj, best_r2 = None, -float("inf")

# 4a. Ridge + Random Forest: raw target, scaled features
for name, model in [
    ("Ridge Baseline", Ridge(alpha=1.0)),
    ("Random Forest", RandomForestRegressor(
        n_estimators=300, max_depth=6, min_samples_leaf=25,
        random_state=42, n_jobs=-1)),
]:
    t0 = time.time()
    model.fit(X_train_scaled, train_df[target_col])
    elapsed = round(time.time() - t0, 2)

    pred_test = np.maximum(0, model.predict(X_test_scaled))
    pred_train = np.maximum(0, model.predict(X_train_scaled))

    tr_r2   = round(r2_score(train_df[target_col], pred_train), 4)
    te_r2   = round(r2_score(y_test_raw, pred_test), 4)
    te_rmse = round(float(np.sqrt(mean_squared_error(y_test_raw, pred_test))), 2)
    te_mae  = round(float(mean_absolute_error(y_test_raw, pred_test)), 2)

    fname = f"{name.lower().replace(' ', '_')}_model.pkl"
    joblib.dump(model, os.path.join("pkl_models", fname))
    if te_r2 > best_r2: best_r2 = te_r2; best_model_obj = model

    results.append({
        "model": name, "train_r2": tr_r2, "test_r2": te_r2,
        "train_rmse": None, "test_rmse": te_rmse, "test_mae": te_mae,
        "test_mape": None, "training_seconds": elapsed, "model_path": fname
    })
    print(f"{name:20s}  train_R2={tr_r2:.4f}  test_R2={te_r2:.4f}  RMSE={te_rmse:.0f}  [{elapsed}s]  [raw+scaled]")

# Save shared scaler for Ridge+RF inference
joblib.dump(scaler_raw, os.path.join("pkl_models", "raw_target_scaler.pkl"))
# Also keep legacy name for backwards compat
joblib.dump(scaler_raw, os.path.join("pkl_models", "ridge_scaler.pkl"))

# 4b. XGBoost + LightGBM: log target (gradient boosting handles log space better)
for name, model in [
    ("XGBoost", xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.02, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
        reg_alpha=0.1, reg_lambda=5.0, random_state=42, n_jobs=-1
    )),
    ("LightGBM", lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.02, num_leaves=20,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=5.0, random_state=42, verbose=-1
    )),
]:
    t0 = time.time()
    model.fit(X_train, y_train_log)
    elapsed = round(time.time() - t0, 2)

    y_pred_raw = np.expm1(np.maximum(0, model.predict(X_test)))
    y_tr_raw   = np.expm1(np.maximum(0, model.predict(X_train)))

    tr_r2   = round(r2_score(np.expm1(y_train_log), y_tr_raw), 4)
    te_r2   = round(r2_score(y_test_raw, y_pred_raw), 4)
    te_rmse = round(float(np.sqrt(mean_squared_error(y_test_raw, y_pred_raw))), 2)
    te_mae  = round(float(mean_absolute_error(y_test_raw, y_pred_raw)), 2)

    fname = f"{name.lower().replace(' ', '_')}_model.pkl"
    joblib.dump(model, os.path.join("pkl_models", fname))
    if te_r2 > best_r2: best_r2 = te_r2; best_model_obj = model

    results.append({
        "model": name, "train_r2": tr_r2, "test_r2": te_r2,
        "train_rmse": None, "test_rmse": te_rmse, "test_mae": te_mae,
        "test_mape": None, "training_seconds": elapsed, "model_path": fname
    })
    print(f"{name:20s}  train_R2={tr_r2:.4f}  test_R2={te_r2:.4f}  RMSE={te_rmse:.0f}  [{elapsed}s]  [log target]")

if best_model_obj:
    joblib.dump(best_model_obj, os.path.join("pkl_models", "best_model.pkl"))

# 5. Save state stats + feature metadata
state_stats.to_csv(os.path.join("pkl_models", "state_stats.csv"), index=False)
with open(os.path.join("pkl_models", "feature_metadata.json"), "w") as f:
    json.dump({
        "feature_cols": feature_cols,
        "log_target": True,
        "raw_target_models": ["Ridge Baseline", "Random Forest"],
        "log_target_models": ["XGBoost", "LightGBM"],
        "split_date": str(pd.Timestamp(split_date).date())
    }, f, indent=2)

# 6. Upsert into model_comparison.json (keep LSTM entry)
comp_path = os.path.join("pkl_models", "model_comparison.json")
existing  = json.load(open(comp_path)) if os.path.exists(comp_path) else []
lstm_only = [e for e in existing if e.get("model") == "LSTM (PyTorch)"]
with open(comp_path, "w") as f:
    json.dump(lstm_only + results, f, indent=2)

results_df = pd.DataFrame(results).sort_values("test_r2", ascending=False)
print()
print(results_df[["model","train_r2","test_r2","test_rmse","test_mae"]].to_string(index=False))


Train: up to 2025-10-31 | Test: after
Train rows: 12688 | Test rows: 3224
Feature count: 41


Ridge Baseline        train_R2=0.2568  test_R2=0.4182  RMSE=1394  [0.02s]  [raw+scaled]


Random Forest         train_R2=0.3213  test_R2=0.1790  RMSE=1656  [5.69s]  [raw+scaled]


XGBoost               train_R2=0.5404  test_R2=0.2506  RMSE=1582  [2.84s]  [log target]


LightGBM              train_R2=0.6288  test_R2=0.2632  RMSE=1569  [7.19s]  [log target]

         model  train_r2  test_r2  test_rmse  test_mae
Ridge Baseline    0.2568   0.4182    1394.10    573.07
      LightGBM    0.6288   0.2632    1568.80    614.20
       XGBoost    0.5404   0.2506    1582.17    540.72
 Random Forest    0.3213   0.1790    1656.04    605.33
